In [5]:
import pandas as pd

#read csv file
df = pd.read_csv('/content/Sample - Superstore.csv', encoding='windows-1252')
print(df.head(2))

   Row ID        Order ID Order Date   Ship Date     Ship Mode Customer ID  \
0       1  CA-2016-152156  11/8/2016  11/11/2016  Second Class    CG-12520   
1       2  CA-2016-152156  11/8/2016  11/11/2016  Second Class    CG-12520   

  Customer Name   Segment        Country       City  ... Postal Code  Region  \
0   Claire Gute  Consumer  United States  Henderson  ...       42420   South   
1   Claire Gute  Consumer  United States  Henderson  ...       42420   South   

        Product ID   Category Sub-Category  \
0  FUR-BO-10001798  Furniture    Bookcases   
1  FUR-CH-10000454  Furniture       Chairs   

                                        Product Name   Sales  Quantity  \
0                  Bush Somerset Collection Bookcase  261.96         2   
1  Hon Deluxe Fabric Upholstered Stacking Chairs,...  731.94         3   

   Discount    Profit  
0       0.0   41.9136  
1       0.0  219.5820  

[2 rows x 21 columns]


In [8]:
import os
import sqlite3
import pandas as pd



conn = sqlite3.connect('superstore.db')

# clean spces and other things
df.columns = [c.lower().replace(' ','_').replace('-','_') for c in df.columns]

# save dataframe to new database
df.to_sql('superstore_raw',conn,if_exists='replace',index=False)

print("successfully created the supersotre_raw database")

successfully created the supersotre_raw database


In [9]:

query = "SELECT * FROM superstore_raw limit 5;"

#query result
pd.read_sql_query(query, conn)

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [12]:
#create the customer table
conn.execute("""
CREATE TABLE if not exists customers AS
 SELECT DISTINCT customer_id, customer_name, segment
FROM  superstore_raw;
""")

# Create the product table
conn.execute("""
CREATE TABLE if not exists products AS
  SELECT DISTINCT product_id, product_name, category, sub_category
FROM superstore_raw;
""")

# Create the order table
conn.execute("""
CREATE TABLE if not exists orders AS
SELECT DISTINCT order_id, customer_id, product_id, order_date, sales, quantity, discount, profit
FROM superstore_raw;
""")

print("customer ,product and order tables have been created")

customer ,product and order tables have been created


In [13]:
# 1. Preview the Customers table
print("CUSTOMERS TABLE PREVIEW:")
display(pd.read_sql_query("SELECT * from customers LIMIT 3;", conn))
print("\n")

# 2. Preview the Products table
print("PRODUCTS TABLE PREVIEW:")
display(pd.read_sql_query("SELECT * from products LIMIT 3;", conn))
print("\n")

# 3. Preview the Orders table
print("ORDERS TABLE PREVIEW:")
display(pd.read_sql_query("SELECT * from orders LIMIT 3;", conn))

CUSTOMERS TABLE PREVIEW:


,customer_id,customer_name,segment
0,CG-12520,Claire Gute,Consumer
1,DV-13045,Darrin Van Huff,Corporate
2,SO-20335,Sean O'Donnell,Consumer




PRODUCTS TABLE PREVIEW:


,product_id,product_name,category,sub_category
0,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases
1,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels




ORDERS TABLE PREVIEW:


,order_id,customer_id,product_id,order_date,sales,quantity,discount,profit
0,CA-2016-152156,CG-12520,FUR-BO-10001798,11/8/2016,261.96,2,0.0,41.9136
1,CA-2016-152156,CG-12520,FUR-CH-10000454,11/8/2016,731.94,3,0.0,219.5820
2,CA-2016-138688,DV-13045,OFF-LA-10000240,6/12/2016,14.62,2,0.0,6.8714


In [14]:
#query 1:	Find all orders where sales are greater than the average sales. (Subquery)
query1="""
select * from orders
where sales > (select avg(sales) from orders);
"""
pd.read_sql_query(query1,conn)

,order_id,customer_id,product_id,order_date,sales,quantity,discount,profit
0,CA-2016-152156,CG-12520,FUR-BO-10001798,11/8/2016,261.9600,2,0.00,41.9136
1,CA-2016-152156,CG-12520,FUR-CH-10000454,11/8/2016,731.9400,3,0.00,219.5820
2,US-2015-108966,SO-20335,FUR-TA-10000577,10/11/2015,957.5775,5,0.45,-383.0310
3,CA-2014-115812,BH-11710,TEC-PH-10002275,6/9/2014,907.1520,6,0.20,90.7152
4,CA-2014-115812,BH-11710,FUR-TA-10001539,6/9/2014,1706.1840,9,0.20,85.3092
...,...,...,...,...,...,...,...,...
2354,US-2016-103674,AP-10720,TEC-PH-10004080,12/6/2016,271.9600,5,0.20,27.1960
2355,US-2016-103674,AP-10720,TEC-PH-10002496,12/6/2016,249.5840,2,0.20,31.1980
2356,US-2016-103674,AP-10720,OFF-BI-10002026,12/6/2016,437.4720,14,0.20,153.1152
2357,CA-2017-121258,DB-13060,TEC-PH-10003645,2/26/2017,258.5760,2,0.20,19.3932


In [15]:
#query 2:Find the highest sales order for each customer.(Subquery)
query_2 = """
SELECT
customer_id,order_id,sales
FROM orders o
WHERE sales=(
    SELECT MAX(sales)
    FROM orders sub_o
    WHERE sub_o.customer_id = o.customer_id);
"""

pd.read_sql_query(query_2, conn)

,customer_id,order_id,sales
0,CG-12520,CA-2016-152156,731.9400
1,SO-20335,US-2015-108966,957.5775
2,BH-11710,CA-2014-115812,1706.1840
3,EB-13870,CA-2015-106320,1044.6300
4,TB-21520,US-2015-150630,3083.4300
...,...,...,...
790,DH-13075,CA-2015-159534,1087.9360
791,IM-15055,CA-2016-129630,2799.9600
792,HW-14935,CA-2017-121559,2405.2000
793,RB-19435,CA-2017-153871,735.9800


In [16]:
#query 3:Calculate total sales for each customer.(CTE)
query_3 = """
WITH CustomerSales AS (
  SELECT
  customer_id,SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
c.customer_name,cs.customer_id,cs.total_sales
FROM CustomerSales cs
JOIN customers c ON cs.customer_id = c.customer_id
ORDER BY cs.total_sales DESC;
"""

# Run the query and display the results
pd.read_sql_query(query_3, conn)

,customer_name,customer_id,total_sales
0,Sean Miller,SM-20320,25043.050
1,Tamara Chand,TC-20980,19052.218
2,Raymond Buch,RB-19360,15117.339
3,Tom Ashbrook,TA-21385,14595.620
4,Adrian Barton,AB-10105,14473.571
...,...,...,...
788,Roy Skaria,RS-19870,22.328
789,Mitch Gastineau,MG-18205,16.739
790,Carl Jackson,CJ-11875,16.520
791,Lela Donovan,LD-16855,5.304


In [17]:
#query 4:Find customers whose total sales are above average.(CTE + Subquery)
query_4 = """
WITH total_sales AS (
    SELECT customer_id, SUM(sales) as total FROM orders GROUP BY customer_id
)
SELECT customer_id, total
FROM total_sales
WHERE total > (SELECT AVG(total) FROM total_sales);
"""
pd.read_sql_query(query_4, conn)

,customer_id,total
0,AA-10315,5563.560
1,AA-10645,5086.935
2,AB-10060,7755.620
3,AB-10105,14473.571
4,AC-10450,5527.846
...,...,...
289,VW-21775,6134.038
290,WB-21850,6160.102
291,YC-21895,5454.350
292,YS-21880,6720.444


In [18]:
#quey 5:Rank all customers based on total sales. (Window Function)
query_5 = """
SELECT customer_id,SUM(sales) AS total,
RANK() OVER (ORDER BY SUM(sales) DESC) AS customer_rank
FROM orders
GROUP BY customer_id;
"""
pd.read_sql_query(query_5, conn)

,customer_id,total,customer_rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3
3,TA-21385,14595.620,4
4,AB-10105,14473.571,5
...,...,...,...
788,RS-19870,22.328,789
789,MG-18205,16.739,790
790,CJ-11875,16.520,791
791,LD-16855,5.304,792


In [20]:
#query 6:Assign row numbers to each order within a customer. (Window Function + PARTITION BY)
query_6 = """
SELECT
    customer_id,order_id,order_date,
ROW_NUMBER() over (partition by customer_id ORDER BY order_date) as row_number
FROM orders;
"""
pd.read_sql_query(query_6,conn)

,customer_id,order_id,order_date,row_number
0,AA-10315,CA-2015-121391,10/4/2015,1
1,AA-10315,CA-2016-103982,3/3/2016,2
2,AA-10315,CA-2016-103982,3/3/2016,3
3,AA-10315,CA-2016-103982,3/3/2016,4
4,AA-10315,CA-2016-103982,3/3/2016,5
...,...,...,...,...
9988,ZD-21925,CA-2016-152471,7/8/2016,5
9989,ZD-21925,CA-2016-152471,7/8/2016,6
9990,ZD-21925,CA-2014-143336,8/27/2014,7
9991,ZD-21925,CA-2014-143336,8/27/2014,8


In [23]:
#query 7:Display top 3 customers based on total sales. (Window Function)
q7_simple = """
select   customer_id,
sum(sales) as total
from orders
    group by   customer_id
   order by total   desc
   limit 3;
"""
pd.read_sql_query(q7_simple, conn)

,customer_id,total
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339


In [24]:
#Write one final query that shows:•	Customer Name •	Total Sales •	Rank (Use JOIN + CTE + Window Function together)
q_f = """
with customer_totals as (
   select customer_id,
sum(sales) as total_sales
   from orders
   group by customer_id
)
select
   c.customer_name,ct.total_sales,
   rank() over (order by ct.total_sales desc) as customer_rank
from customer_totals ct join customers c   on ct.customer_id = c.customer_id;
"""
pd.read_sql_query(q_f, conn)


,customer_name,total_sales,customer_rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
...,...,...,...
788,Roy Skaria,22.328,789
789,Mitch Gastineau,16.739,790
790,Carl Jackson,16.520,791
791,Lela Donovan,5.304,792


SUMMARY OF THE **PROJECT**

In [25]:
#1->Who are the top 5 customers?
q_1 = """
select customer_id,sum(sales) as total
from orders group by customer_id
order by total   desc
limit 5;
"""
pd.read_sql_query(q_1,conn)

,customer_id,total
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571


In [26]:
#2->who are bottom 5 customer?
q_2="""
select customer_id,sum(sales) as total
from orders group by customer_id
order by total   asc
limit 5;
"""
pd.read_sql_query(q_2,conn)

,customer_id,total
0,TS-21085,4.833
1,LD-16855,5.304
2,CJ-11875,16.520
3,MG-18205,16.739
4,RS-19870,22.328


In [28]:
#3->Which customers made only one order?
q_3= """
select customer_id,count(distinct order_id) as order_count
from orders group by customer_id
having order_count = 1;
"""
pd.read_sql_query(q_3,conn)

,customer_id,order_count
0,AO-10810,1
1,AR-10570,1
2,CJ-11875,1
3,JC-15385,1
4,JR-15700,1
5,LD-16855,1
6,MG-18205,1
7,PH-18790,1
8,RE-19405,1
9,RM-19750,1


In [29]:
#4->Which customers have above-average sales?
q_4="""
select customer_id,sum(sales) as total
from orders group by customer_id
having total >(select sum(sales)/count(distinct customer_id) from orders);
"""
pd.read_sql_query(q_4,conn)

,customer_id,total
0,AA-10315,5563.560
1,AA-10645,5086.935
2,AB-10060,7755.620
3,AB-10105,14473.571
4,AC-10450,5527.846
...,...,...
289,VW-21775,6134.038
290,WB-21850,6160.102
291,YC-21895,5454.350
292,YS-21880,6720.444


In [31]:
#5->What is the highest order value per customer?
q_5="""
select customer_id,max(sales) as max_val
from orders group by customer_id;
"""
pd.read_sql_query(q_5,conn)

,customer_id,max_val
0,AA-10315,3930.072
1,AA-10375,499.980
2,AA-10480,479.970
3,AA-10645,1323.900
4,AB-10015,341.960
...,...,...
788,XP-21865,337.088
789,YC-21895,2934.330
790,YS-21880,2793.528
791,ZC-21910,1516.200
